# CRISPRon — LuoSpCas92020_min200 — 100-trial Optuna training

Run the corresponding preprocessing/splitting notebook first:

```text
CRISPRon_LuoSpCas92020_min200_10543_complete_fixed.ipynb
```

This training notebook expects the saved split under:

```text
results/crispron_LuoSpCas92020_min200_fixed_split/
```

and uses:

- `30mer_gRNA` as the 30-nt sequence input;
- `Quant_norm_efficiency` as the activity label;
- `CRISPRoff` as the scalar thermodynamic input.

The high-level CRISPRon `TYPE=CG` topology is preserved:

1. 30-nt one-hot sequence input;
2. three parallel Conv1D → Dropout → AveragePooling1D → Flatten branches;
3. concatenate the three branches;
4. Dense → Dropout;
5. concatenate the scalar CRISPRoff input;
6. Dense → Dropout;
7. Dense → Dropout;
8. one linear regression output.

Adam remains the only optimizer and MSE remains the loss.

Optuna tunes the detailed architectural/training hyperparameters while the
fixed 64/16/20 split remains unchanged.


In [ ]:
from pathlib import Path
import hashlib
import json
import random
import warnings

import numpy as np
import optuna
import pandas as pd
import tensorflow as tf

from scipy.stats import pearsonr, spearmanr
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import (
    AveragePooling1D,
    Conv1D,
    Dense,
    Dropout,
    Flatten,
    concatenate,
)

OUTPUT_DIR = Path(
    "results/crispron_LuoSpCas92020_min200_fixed_split"
)

SPLIT_DIR = OUTPUT_DIR / "saved_splits"
RESULTS_DIR = OUTPUT_DIR / "trial_results"

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAIN_FILE = (
    SPLIT_DIR / "train_split.csv"
)

VALIDATION_FILE = (
    SPLIT_DIR / "validation_split.csv"
)

UNSEEN_FILE = (
    SPLIT_DIR / "unseen_split.csv"
)

MANIFEST_FILE = (
    OUTPUT_DIR / "split_manifest.json"
)

N_TRIALS = 100

MODEL_SEED = 42
OPTUNA_SEED = 42

MAX_EPOCHS = 100
PATIENCE = 10

for path in [
    TRAIN_FILE,
    VALIDATION_FILE,
    UNSEEN_FILE,
    MANIFEST_FILE,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path.resolve()}\n"
            "Run the LuoSpCas92020_min200 preprocessing/splitting notebook first."
        )

print("TensorFlow:", tf.__version__)
print("Optuna:", optuna.__version__)
print("Output directory:", OUTPUT_DIR.resolve())


## Load and verify the fixed Luo-only split

The manifest hash checks ensure this notebook is training on the exact split
created by the preprocessing notebook.


In [ ]:
def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


with open(
    MANIFEST_FILE,
    "r",
) as handle:
    manifest = json.load(handle)

assert (
    sha256(TRAIN_FILE)
    == manifest["train_sha256"]
)

assert (
    sha256(VALIDATION_FILE)
    == manifest["validation_sha256"]
)

assert (
    sha256(UNSEEN_FILE)
    == manifest["unseen_sha256"]
)

train_data = pd.read_csv(
    TRAIN_FILE
)

validation_data = pd.read_csv(
    VALIDATION_FILE
)

unseen_data = pd.read_csv(
    UNSEEN_FILE
)

required_columns = {
    "30mer_gRNA",
    "Quant_norm_efficiency",
    "CRISPRoff",
}

for subset_name, frame in [
    ("train", train_data),
    ("validation", validation_data),
    ("unseen", unseen_data),
]:
    missing = required_columns.difference(
        frame.columns
    )

    if missing:
        raise ValueError(
            f"{subset_name} is missing columns: "
            f"{sorted(missing)}"
        )

    valid_sequences = (
        frame["30mer_gRNA"]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(
            "U",
            "T",
            regex=False,
        )
        .str.fullmatch(r"[ACGT]{30}")
    )

    if not valid_sequences.all():
        raise ValueError(
            f"{subset_name} contains invalid 30-nt sequences."
        )

print("Selected dataset:", manifest.get(
    "selected_dataset",
    "LuoSpCas92020_min200",
))

print(
    "Subset sizes:",
    len(train_data),
    len(validation_data),
    len(unseen_data),
)

print(
    "Total:",
    (
        len(train_data)
        + len(validation_data)
        + len(unseen_data)
    ),
)


## CRISPRon input preprocessing

The 30-mer is one-hot encoded into four channels:

```text
A = [1, 0, 0, 0]
C = [0, 1, 0, 0]
G = [0, 0, 1, 0]
T = [0, 0, 0, 1]
```

The CRISPRoff score remains a separate scalar input.


In [ ]:
BASE_TO_INDEX = {
    "A": 0,
    "C": 1,
    "G": 2,
    "T": 3,
}


def encode_30mers(series):
    sequences = (
        series.astype(str)
        .str.strip()
        .str.upper()
        .str.replace(
            "U",
            "T",
            regex=False,
        )
        .tolist()
    )

    encoded = np.zeros(
        (
            len(sequences),
            30,
            4,
        ),
        dtype=np.float32,
    )

    for row_index, sequence in enumerate(
        sequences
    ):
        if (
            len(sequence) != 30
            or any(
                base not in BASE_TO_INDEX
                for base in sequence
            )
        ):
            raise ValueError(
                f"Invalid 30-mer: {sequence}"
            )

        for position, base in enumerate(
            sequence
        ):
            encoded[
                row_index,
                position,
                BASE_TO_INDEX[base],
            ] = 1.0

    return encoded


X_train_seq = encode_30mers(
    train_data["30mer_gRNA"]
)

X_validation_seq = encode_30mers(
    validation_data["30mer_gRNA"]
)

X_unseen_seq = encode_30mers(
    unseen_data["30mer_gRNA"]
)


X_train_g = (
    train_data["CRISPRoff"]
    .to_numpy(
        dtype=np.float32
    )
    .reshape(-1, 1)
)

X_validation_g = (
    validation_data["CRISPRoff"]
    .to_numpy(
        dtype=np.float32
    )
    .reshape(-1, 1)
)

X_unseen_g = (
    unseen_data["CRISPRoff"]
    .to_numpy(
        dtype=np.float32
    )
    .reshape(-1, 1)
)


y_train = (
    train_data[
        "Quant_norm_efficiency"
    ]
    .to_numpy(
        dtype=np.float32
    )
)

y_validation = (
    validation_data[
        "Quant_norm_efficiency"
    ]
    .to_numpy(
        dtype=np.float32
    )
)

y_unseen = (
    unseen_data[
        "Quant_norm_efficiency"
    ]
    .to_numpy(
        dtype=np.float32
    )
)

print(
    "Sequence:",
    X_train_seq.shape,
)

print(
    "CRISPRoff:",
    X_train_g.shape,
)

print(
    "Labels:",
    y_train.shape,
)

print(
    "Label range:",
    float(y_train.min()),
    "to",
    float(y_train.max()),
)


## CRISPRon high-level topology

The topology remains unchanged across trials. Optuna tunes detailed filter
counts, kernel sizes, pooling sizes, dropout rates, dense widths, Adam learning
rate, and batch size.


In [ ]:
def build_crispron_model(
    params
):
    input_sequence = Input(
        shape=(30, 4),
        name="input_onehot",
    )

    input_crisproff = Input(
        shape=(1,),
        name="input_dGB",
    )

    branch_outputs = []

    for branch in [
        1,
        2,
        3,
    ]:
        x = Conv1D(
            filters=params[
                f"conv{branch}_filters"
            ],
            kernel_size=params[
                f"conv{branch}_kernel"
            ],
            activation="relu",
            padding="valid",
            name=f"conv_{branch}",
        )(
            input_sequence
        )

        x = Dropout(
            params[
                f"conv{branch}_dropout"
            ],
            name=f"drop_{branch}",
        )(x)

        x = AveragePooling1D(
            pool_size=params[
                f"conv{branch}_pool"
            ],
            padding="same",
            name=f"pool_{branch}",
        )(x)

        x = Flatten(
            name=f"flatten_{branch}",
        )(x)

        branch_outputs.append(
            x
        )

    x = concatenate(
        branch_outputs,
        name="concat_conv",
    )

    x = Dense(
        params[
            "dense0_units"
        ],
        activation="relu",
        name="dense_0",
    )(x)

    x = Dropout(
        params[
            "dense0_dropout"
        ],
        name="drop_d0",
    )(x)

    # Original TYPE=CG CRISPRoff fusion location.
    x = concatenate(
        [
            x,
            input_crisproff,
        ],
        name="concat_crisproff",
    )

    x = Dense(
        params[
            "dense1_units"
        ],
        activation="relu",
        name="dense_1",
    )(x)

    x = Dropout(
        params[
            "dense1_dropout"
        ],
        name="drop_d1",
    )(x)

    x = Dense(
        params[
            "dense2_units"
        ],
        activation="relu",
        name="dense_2",
    )(x)

    x = Dropout(
        params[
            "dense2_dropout"
        ],
        name="drop_d2",
    )(x)

    output = Dense(
        1,
        activation=None,
        name="output",
    )(x)

    model = Model(
        inputs=[
            input_sequence,
            input_crisproff,
        ],
        outputs=output,
        name="CRISPRon_tunable",
    )

    # Adam remains the only optimizer.
    try:
        optimizer = (
            tf.keras.optimizers.legacy.Adam(
                learning_rate=params[
                    "learning_rate"
                ]
            )
        )
    except Exception:
        optimizer = (
            tf.keras.optimizers.Adam(
                learning_rate=params[
                    "learning_rate"
                ]
            )
        )

    model.compile(
        optimizer=optimizer,
        loss="mse",
        metrics=[
            "mae",
            "mse",
        ],
    )

    return model


## Reproducibility and metrics


In [ ]:
def set_all_seeds(
    seed
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    tf.random.set_seed(
        seed
    )


def safe_pearson(
    y_true,
    y_pred,
):
    if (
        len(y_true) < 2
        or np.std(y_true) == 0
        or np.std(y_pred) == 0
    ):
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore"
        )

        return float(
            pearsonr(
                y_true,
                y_pred,
            )[0]
        )


def safe_spearman(
    y_true,
    y_pred,
):
    if (
        len(y_true) < 2
        or np.std(y_true) == 0
        or np.std(y_pred) == 0
    ):
        return np.nan

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore"
        )

        return float(
            spearmanr(
                y_true,
                y_pred,
            )[0]
        )


## 100 Optuna trials

The high-level CRISPRon model remains fixed, while the following are tuned:

- three convolution filter counts
- three kernel sizes
- three pooling sizes
- three branch dropout rates
- three dense widths
- three dense dropout rates
- Adam learning rate
- batch size

The objective is **validation MSE**.

Unseen Pearson and Spearman are recorded but are not used by Optuna for model
selection.


In [ ]:
trial_records = []

best_validation_mse = (
    np.inf
)

best_trial_number = (
    None
)

best_weights_file = (
    RESULTS_DIR
    / "best_CRISPRon.weights.h5"
)


def suggest_parameters(
    trial
):
    return {
        "conv1_filters":
            trial.suggest_int(
                "conv1_filters",
                48,
                160,
                step=16,
            ),

        "conv1_kernel":
            trial.suggest_categorical(
                "conv1_kernel",
                [
                    2,
                    3,
                    4,
                ],
            ),

        "conv1_dropout":
            trial.suggest_float(
                "conv1_dropout",
                0.10,
                0.50,
            ),

        "conv1_pool":
            trial.suggest_categorical(
                "conv1_pool",
                [
                    2,
                    3,
                ],
            ),

        "conv2_filters":
            trial.suggest_int(
                "conv2_filters",
                32,
                128,
                step=16,
            ),

        "conv2_kernel":
            trial.suggest_categorical(
                "conv2_kernel",
                [
                    4,
                    5,
                    6,
                ],
            ),

        "conv2_dropout":
            trial.suggest_float(
                "conv2_dropout",
                0.10,
                0.50,
            ),

        "conv2_pool":
            trial.suggest_categorical(
                "conv2_pool",
                [
                    2,
                    3,
                ],
            ),

        "conv3_filters":
            trial.suggest_int(
                "conv3_filters",
                16,
                96,
                step=16,
            ),

        "conv3_kernel":
            trial.suggest_categorical(
                "conv3_kernel",
                [
                    6,
                    7,
                    8,
                ],
            ),

        "conv3_dropout":
            trial.suggest_float(
                "conv3_dropout",
                0.10,
                0.50,
            ),

        "conv3_pool":
            trial.suggest_categorical(
                "conv3_pool",
                [
                    2,
                    3,
                ],
            ),

        "dense0_units":
            trial.suggest_int(
                "dense0_units",
                48,
                160,
                step=16,
            ),

        "dense0_dropout":
            trial.suggest_float(
                "dense0_dropout",
                0.10,
                0.50,
            ),

        "dense1_units":
            trial.suggest_int(
                "dense1_units",
                48,
                160,
                step=16,
            ),

        "dense1_dropout":
            trial.suggest_float(
                "dense1_dropout",
                0.10,
                0.50,
            ),

        "dense2_units":
            trial.suggest_int(
                "dense2_units",
                32,
                128,
                step=16,
            ),

        "dense2_dropout":
            trial.suggest_float(
                "dense2_dropout",
                0.10,
                0.50,
            ),

        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                1e-5,
                3e-3,
                log=True,
            ),

        "batch_size":
            trial.suggest_categorical(
                "batch_size",
                [
                    32,
                    64,
                    128,
                    256,
                    500,
                ],
            ),
    }


def objective(
    trial
):
    global best_validation_mse
    global best_trial_number

    tf.keras.backend.clear_session()

    set_all_seeds(
        MODEL_SEED
    )

    params = (
        suggest_parameters(
            trial
        )
    )

    model = (
        build_crispron_model(
            params
        )
    )

    early_stopping = (
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=0,
        )
    )

    history = model.fit(
        [
            X_train_seq,
            X_train_g,
        ],
        y_train,
        validation_data=(
            [
                X_validation_seq,
                X_validation_g,
            ],
            y_validation,
        ),
        epochs=MAX_EPOCHS,
        batch_size=params[
            "batch_size"
        ],
        shuffle=True,
        verbose=0,
        callbacks=[
            early_stopping
        ],
    )

    validation_losses = (
        np.asarray(
            history.history[
                "val_loss"
            ],
            dtype=float,
        )
    )

    best_epoch = (
        int(
            np.argmin(
                validation_losses
            )
        )
        + 1
    )

    validation_prediction = (
        model.predict(
            [
                X_validation_seq,
                X_validation_g,
            ],
            verbose=0,
        )
        .reshape(-1)
    )

    validation_mse = float(
        np.mean(
            (
                y_validation
                - validation_prediction
            )
            ** 2
        )
    )

    unseen_prediction = (
        model.predict(
            [
                X_unseen_seq,
                X_unseen_g,
            ],
            verbose=0,
        )
        .reshape(-1)
    )

    unseen_pearson = (
        safe_pearson(
            y_unseen,
            unseen_prediction,
        )
    )

    unseen_spearman = (
        safe_spearman(
            y_unseen,
            unseen_prediction,
        )
    )

    trial.set_user_attr(
        "best_epoch",
        best_epoch,
    )

    trial_records.append(
        {
            "trial":
                trial.number,

            "validation_mse":
                validation_mse,

            "unseen_pearson":
                unseen_pearson,

            "unseen_spearman":
                unseen_spearman,

            "best_epoch":
                best_epoch,

            **params,
        }
    )

    if (
        validation_mse
        < best_validation_mse
    ):
        best_validation_mse = (
            validation_mse
        )

        best_trial_number = (
            trial.number
        )

        model.save_weights(
            best_weights_file
        )

    print(
        f"Trial {trial.number:3d} | "
        f"Validation MSE="
        f"{validation_mse:.8f} | "
        f"Unseen Pearson="
        f"{unseen_pearson:.4f} | "
        f"Unseen Spearman="
        f"{unseen_spearman:.4f} | "
        f"Epoch={best_epoch}"
    )

    return validation_mse


study = (
    optuna.create_study(
        direction="minimize",
        sampler=(
            optuna.samplers.TPESampler(
                seed=OPTUNA_SEED
            )
        ),
        study_name=(
            "CRISPRon_"
            "LuoSpCas92020_min200_"
            "100_trials"
        ),
    )
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    gc_after_trial=True,
)


## Save all trial metrics and the best model


In [ ]:
results_df = (
    pd.DataFrame(
        trial_records
    )
    .sort_values(
        "trial"
    )
    .reset_index(
        drop=True
    )
)

results_df.to_csv(
    RESULTS_DIR
    / "all_100_trial_metrics_and_parameters.csv",
    index=False,
)

results_df[
    "validation_mse"
].to_csv(
    RESULTS_DIR
    / "Validation_loss.txt",
    index=False,
    header=False,
)

results_df[
    "unseen_pearson"
].to_csv(
    RESULTS_DIR
    / "Unseen_Pearson.txt",
    index=False,
    header=False,
)

results_df[
    "unseen_spearman"
].to_csv(
    RESULTS_DIR
    / "Unseen_Spearman.txt",
    index=False,
    header=False,
)

if (
    best_trial_number
    != study.best_trial.number
):
    raise RuntimeError(
        "Saved best weights do not match Optuna's best trial."
    )

best_params = dict(
    study.best_trial.params
)

best_model = (
    build_crispron_model(
        best_params
    )
)

best_model.load_weights(
    best_weights_file
)

best_model.save(
    RESULTS_DIR
    / "best_CRISPRon.keras"
)

best_row = (
    results_df.loc[
        results_df["trial"]
        == study.best_trial.number
    ]
    .iloc[0]
)

summary = {
    "dataset":
        "LuoSpCas92020_min200",

    "expected_total_samples":
        10543,

    "sequence_column":
        "30mer_gRNA",

    "activity_column":
        "Quant_norm_efficiency",

    "additional_input":
        "CRISPRoff",

    "type":
        "CG",

    "high_level_architecture_modified":
        False,

    "detailed_architecture_tuned":
        True,

    "preprocessing_modified":
        False,

    "optimizer":
        "Adam",

    "loss":
        "mse",

    "n_trials":
        N_TRIALS,

    "max_epochs":
        MAX_EPOCHS,

    "patience":
        PATIENCE,

    "model_selection_metric":
        "validation_mse",

    "best_trial":
        int(
            study.best_trial.number
        ),

    "best_params":
        best_params,

    "best_epoch":
        int(
            study.best_trial.user_attrs[
                "best_epoch"
            ]
        ),

    "validation_mse":
        float(
            best_row[
                "validation_mse"
            ]
        ),

    "unseen_pearson":
        float(
            best_row[
                "unseen_pearson"
            ]
        ),

    "unseen_spearman":
        float(
            best_row[
                "unseen_spearman"
            ]
        ),
}

with open(
    RESULTS_DIR
    / "best_trial_summary.json",
    "w",
) as handle:
    json.dump(
        summary,
        handle,
        indent=2,
    )

print(
    "Best trial:",
    study.best_trial.number,
)

print(
    "Best parameters:"
)

print(
    json.dumps(
        best_params,
        indent=2,
    )
)

print(
    "Best validation MSE:",
    float(
        best_row[
            "validation_mse"
        ]
    ),
)

print(
    "Best unseen Pearson:",
    float(
        best_row[
            "unseen_pearson"
        ]
    ),
)

print(
    "Best unseen Spearman:",
    float(
        best_row[
            "unseen_spearman"
        ]
    ),
)

print(
    "Saved to:",
    RESULTS_DIR.resolve(),
)
